In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from jppype import vscode_theme

from fundus_vessels_toolkit.models.topology.dataset import BranchDigraphDataset
from fundus_vessels_toolkit.models.topology.losses import BranchDigraphMiner
from fundus_vessels_toolkit.models.topology.model import BranchDigraphModel
from fundus_vessels_toolkit.segment_to_graph.vbranch_digraph import VBranchDigraph
from train import DigraphGNNTrainer, DigraphGNNTrainerConfig

vscode_theme()


HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [3]:
dataset = BranchDigraphDataset("ALL_DATA_bundle.tar.gz")
train_set, val_set, test_set = dataset.split_sets(train_ratio=0.7, val_ratio=0.15)

Processing...
Done!


## Visualize result from pred table


In [4]:
def parse_arborescence(preds, idx, opti=False):
    pred = preds.loc[idx] if isinstance(idx, str) else preds.iloc[idx]
    b_parent = np.fromstring(pred[("opti_" if opti else "") + "parent"], sep=",", dtype=int)
    b_dir = np.fromstring(pred[("opti_" if opti else "") + "dir"], sep=",", dtype=int)
    b_av = np.fromstring(pred["av"], sep=",", dtype=int)
    return pred.name, b_parent, b_dir, b_av


### Load model from checkpoint


In [ ]:
import torch

torch.load("GNN-Topo-v1/c2kx8j5h/checkpoints/epoch=239-step=8880.ckpt")["state_dict"].keys()

odict_keys(['model.img_feature_extractor.efficient_net.0.0.weight', 'model.img_feature_extractor.efficient_net.0.1.weight', 'model.img_feature_extractor.efficient_net.0.1.bias', 'model.img_feature_extractor.efficient_net.0.1.running_mean', 'model.img_feature_extractor.efficient_net.0.1.running_var', 'model.img_feature_extractor.efficient_net.0.1.num_batches_tracked', 'model.img_feature_extractor.efficient_net.1.0.block.0.0.weight', 'model.img_feature_extractor.efficient_net.1.0.block.0.1.weight', 'model.img_feature_extractor.efficient_net.1.0.block.0.1.bias', 'model.img_feature_extractor.efficient_net.1.0.block.0.1.running_mean', 'model.img_feature_extractor.efficient_net.1.0.block.0.1.running_var', 'model.img_feature_extractor.efficient_net.1.0.block.0.1.num_batches_tracked', 'model.img_feature_extractor.efficient_net.1.1.block.0.0.weight', 'model.img_feature_extractor.efficient_net.1.1.block.0.1.weight', 'model.img_feature_extractor.efficient_net.1.1.block.0.1.bias', 'model.img_featu

In [ ]:
model = (
    BranchDigraphModel.load_state_dict(torch.load("GNN-Topo-v1/c2kx8j5h/checkpoints/epoch=239-step=8880.ckpt"))
    .model.cuda()
    .eval()
)

In [6]:
ID = 4
sample_gt, gt_digraph = val_set.get(ID, version="vascx", return_digraph=True)
with torch.inference_mode():
    out: BranchDigraphModel.Output = model(sample_gt.cuda())
pred_parent, pred_dir = out.max_parent(), out.dir_logit > 0
pred_parent, pred_dir, pred_av = out.optimal_tree

assert VBranchDigraph.has_fp_av_p(gt_digraph)
valid_branch = ~gt_digraph.branch_fp()
av_gt = gt_digraph.branch_av_class() <= 1
print(((out.av_logit.numpy(force=True)[valid_branch] > 0) == av_gt[valid_branch]).mean())
print(((pred_av.numpy(force=True)[valid_branch] > 0) == av_gt[valid_branch]).mean())


0.9373433583959899
0.8521303258145363


In [13]:
m, pred_tree = val_set.show_tree_diff(
    out.name,
    pred_parent.numpy(force=True),
    pred_dir.numpy(force=True),
    out.fp_logit.numpy(force=True) > 0,
    pred_av.numpy(force=True) > 0,
)
m

GridBox(children=(HTML(value='<h3 style="text-align: center;">GT Tree: 074_N</h3>'), HTML(value='<h3 style="te…

In [9]:
STOP

NameError: name 'STOP' is not defined

In [ ]:
np.set_printoptions(linewidth=200, precision=2, suppress=True)
d = out.to_digraph()
d.lines_info(b1=86, sort_by_p=True).round(3).head(20)

,b0,b1,tip0,tip1,line_p,av_p,total_p,b0_dir_p,b1_dir_p
0,16,86,1,0,0.949,0.0,1.451,0.007,0.998
1,129,86,0,1,0.907,0.0,1.366,0.916,0.002
2,38,86,0,0,0.051,0.0,1.047,0.995,0.998
3,-1,86,0,0,0.000,0.0,0.998,0.002,0.998
4,36,86,1,0,0.000,0.0,0.997,0.997,0.998
5,35,86,1,0,0.000,0.0,0.800,0.602,0.998
6,44,86,0,0,0.000,0.0,0.675,0.352,0.998
7,41,86,0,0,0.000,0.0,0.572,0.147,0.998
8,37,86,0,0,0.000,0.0,0.534,0.069,0.998
9,43,86,0,0,0.000,0.0,0.511,0.024,0.998


In [ ]:
import tqdm

from fundus_toolkits import AVLabel
from fundus_toolkits.utils.geometric import Point
from fundus_vessels_toolkit.segment_to_graph.av_tree_parsing import naive_infer_arborescence

name = []
pred_parent_acc = []
pred_dir_acc = []
pred_av_acc = []
opti_parent_acc = []
opti_dir_acc = []
opti_av_acc = []
baseline_parent_acc = []
baseline_dir_acc = []
baseline_av_acc = []

eval_set = val_set

with torch.inference_mode():
    for i in tqdm.tqdm(range(len(eval_set))):
        sample, gt_digraph = eval_set.get(i, version="fvt", return_digraph=True)
        assert VBranchDigraph.has_all_p(gt_digraph) and gt_digraph.graph is not None

        valid_branch = ~gt_digraph.branch_fp()
        av_gt = (gt_digraph.branch_av_class() <= 1)[valid_branch]
        od = Point.parse(sample.od_yx.tolist())

        art_branch = gt_digraph.graph.branch_attr["av"] == AVLabel.ART
        vei_branch = gt_digraph.graph.branch_attr["av"] == AVLabel.VEI
        parent_base = -np.ones(gt_digraph.graph.branch_count, dtype=np.int_)
        dir_base = np.zeros(gt_digraph.graph.branch_count, dtype=np.bool_)
        parent_base[art_branch], dir_base[art_branch] = naive_infer_arborescence(
            gt_digraph.graph, od, branch_subset=art_branch
        )
        parent_base[vei_branch], dir_base[vei_branch] = naive_infer_arborescence(
            gt_digraph.graph, od, branch_subset=vei_branch
        )
        parent_base, dir_base = parent_base[valid_branch], dir_base[valid_branch]
        baseline_parent_acc.append((parent_base == gt_digraph.max_parent()[valid_branch]).mean())
        baseline_dir_acc.append((dir_base == (gt_digraph.branch_dir_p[valid_branch] > 0.5)).mean())
        baseline_av_acc.append((art_branch[valid_branch] == av_gt).mean())

        out = model(sample.cuda())
        name += [out.name]
        pred_parent, pred_dir = out.max_parent(), out.dir_logit > 0

        pred_parent, pred_dir = pred_parent.numpy(force=True), pred_dir.numpy(force=True)
        pred_parent, pred_dir = pred_parent[valid_branch], pred_dir[valid_branch]
        av_logit = out.av_logit.numpy(force=True)[valid_branch]
        pred_parent_acc.append((pred_parent == gt_digraph.max_parent()[valid_branch]).mean())
        pred_dir_acc.append((pred_dir == (gt_digraph.branch_dir_p[valid_branch] > 0.5)).mean())
        pred_av_acc.append(((av_logit > 0) == av_gt).mean())

        opti_parent, opti_dir, opti_av = out.optimal_tree
        opti_parent, opti_dir = opti_parent.numpy(force=True), opti_dir.numpy(force=True)
        opti_parent, opti_dir = opti_parent[valid_branch], opti_dir[valid_branch]
        opti_parent_acc.append((opti_parent == gt_digraph.max_parent()[valid_branch]).mean())
        opti_dir_acc.append((opti_dir == (gt_digraph.branch_dir_p[valid_branch] > 0.5)).mean())

        opti_av = opti_av.numpy(force=True)[valid_branch]
        opti_av_acc.append(((opti_av > 0) == av_gt).mean())

100%|██████████| 53/53 [00:21<00:00,  2.45it/s]


Parent ACC


In [ ]:
np.mean(pred_parent_acc), np.mean(opti_parent_acc), np.mean(baseline_parent_acc)

(np.float64(0.8606960002811269),
 np.float64(0.8733594971206884),
 np.float64(0.8261209401315641))

In [ ]:
np.argsort(pred_parent_acc)

array([42, 41, 38, 34, 43, 27, 29, 23, 28, 51, 32, 16, 33, 52, 30, 37, 25,
       40, 47, 45, 24, 39, 48, 26,  2, 20,  6,  1,  8, 19, 22, 31, 12,  0,
       46, 18, 17,  5, 13, 44,  4, 15, 21,  3,  9, 49, 35, 11, 36, 10,  7,
       50, 14])

Dir ACC


In [ ]:
np.mean(pred_dir_acc), np.mean(opti_dir_acc), np.mean(baseline_dir_acc)

(np.float64(0.9677232103487156),
 np.float64(0.9765069920258027),
 np.float64(0.9359552166526669))

AV ACC


In [ ]:
np.mean(pred_av_acc), np.mean(opti_av_acc), np.mean(baseline_av_acc)

(np.float64(0.9112861514322247),
 np.float64(0.880420397243731),
 np.float64(0.949384537052953))

In [ ]:
np.argsort(pred_av_acc)

array([38, 51, 52, 22, 25, 42, 27, 43, 31, 32, 30, 34,  9, 11, 40,  0,  8,
       37, 13,  4, 29, 12, 23, 16, 17, 24, 39, 18, 26, 21, 48,  3, 20, 45,
        6, 15, 19, 47, 10, 46, 49,  5, 33, 35, 14, 41, 28,  1,  7,  2, 44,
       36, 50])